# Notebook 3 – Porto Seguro Safe Driver Prediction (Classification)

## Overview
We benchmark five tabular-learning methods on the Porto Seguro dataset:
**TabNet**, **FT-Transformer**, **XGBoost**, **LightGBM**, and **Random Forest**.  
Each model is tuned with **Optuna** (20 trials) and evaluated across 3 seeds.  
Metrics: **Accuracy**, **AUC-ROC**, **Normalized Gini** (= 2·AUC − 1), **F1**.

> **Note:** The dataset is heavily imbalanced (~3.6 % positives).  
> If the Kaggle API is not configured, a synthetic dataset matching the Porto
> Seguro schema is generated automatically.


In [ ]:
!pip install pytorch-tabnet "rtdl==0.0.13" optuna xgboost lightgbm ucimlrepo scikit-learn pandas numpy matplotlib seaborn shap

## Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random, os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

import rtdl
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

from pytorch_tabnet.tab_model import TabNetClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## Configuration

In [ ]:
SEEDS           = [42, 123, 456]
N_OPTUNA_TRIALS = 20
TEST_SIZE       = 0.20
VAL_FRAC        = 0.25


## Data Loading

In [ ]:
os.makedirs('data/porto_seguro', exist_ok=True)
df = None

try:
    import kaggle
    kaggle.api.competition_download_files(
        'porto-seguro-safe-driver-prediction', path='data/porto_seguro'
    )
    zip_path = 'data/porto_seguro/porto-seguro-safe-driver-prediction.zip'
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('data/porto_seguro')
    df = pd.read_csv('data/porto_seguro/train.csv')
    print(f"Loaded real data: {df.shape}")
except Exception as e:
    print(f"Kaggle download failed: {e}")
    print("Please manually download train.csv from:")
    print("https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction/data")
    print("and place it at data/porto_seguro/train.csv")
    df = None


In [ ]:
if df is None:
    # Try to load from disk if previously downloaded
    local_path = 'data/porto_seguro/train.csv'
    if os.path.exists(local_path):
        df = pd.read_csv(local_path)
        print(f"Loaded from disk: {df.shape}")


In [ ]:
if df is None:
    print("Creating synthetic demo dataset matching Porto Seguro schema...")
    np.random.seed(42)
    n_rows = 50000

    cols = {}
    cols['id'] = np.arange(n_rows)

    for i in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]:
        cols[f'ps_ind_{i:02d}'] = np.random.randint(0, 5, n_rows)
    for i in [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]:
        cols[f'ps_ind_{i:02d}_bin'] = np.random.randint(0, 2, n_rows)
    cols['ps_ind_02_cat'] = np.random.randint(-1, 5, n_rows)
    cols['ps_ind_04_cat'] = np.random.randint(-1, 3, n_rows)
    cols['ps_ind_05_cat'] = np.random.randint(-1, 7, n_rows)

    for i in [1, 2, 3]:
        cols[f'ps_reg_0{i}'] = np.random.uniform(0, 2, n_rows).astype(np.float32)

    for i in range(1, 16):
        cols[f'ps_car_{i:02d}_cat'] = np.random.randint(-1, 10, n_rows)
    for i in [12, 13, 14, 15]:
        cols[f'ps_car_{i:02d}'] = np.random.uniform(0, 10, n_rows).astype(np.float32)
    cols['ps_car_11'] = np.random.randint(0, 4, n_rows)

    for i in range(1, 21):
        cols[f'ps_calc_{i:02d}'] = np.random.uniform(0, 1, n_rows).astype(np.float32)
    for i in range(1, 21):
        cols[f'ps_calc_{i:02d}_bin'] = np.random.randint(0, 2, n_rows)

    cols['target'] = (np.random.random(n_rows) < 0.036).astype(int)
    df = pd.DataFrame(cols)
    print(f"Synthetic dataset: {df.shape}, positive rate: {df['target'].mean():.3%}")
    print("NOTE: Results on synthetic data are for demonstration only!")


## EDA

In [ ]:
print("Shape:", df.shape)
print()
print("Target distribution:")
print(df['target'].value_counts())
print(f"Positive rate: {df['target'].mean():.3%}")


In [ ]:
# -1 values as missing indicator
n_neg1 = (df == -1).sum()
print("Columns with -1 (missing sentinel):")
print(n_neg1[n_neg1 > 0])


In [ ]:
# Sample correlation heat-map (first 20 cols)
sample_cols = [c for c in df.columns if c not in ['id', 'target']][:20]
plt.figure(figsize=(12, 10))
sns.heatmap(df[sample_cols].corr(), cmap='coolwarm', center=0)
plt.title('Porto Seguro – correlation (first 20 features)')
plt.tight_layout()
plt.show()


## Preprocessing

In [ ]:
df_proc = df.drop(columns=['id']).copy()
y = df_proc['target'].values.astype(np.int64)
df_feat = df_proc.drop(columns=['target'])

cat_cols_ps  = [c for c in df_feat.columns if c.endswith('_cat')]
num_cols_ps  = [c for c in df_feat.columns if c not in cat_cols_ps]

print(f"Numerical features: {len(num_cols_ps)}")
print(f"Categorical features: {len(cat_cols_ps)}")

# Replace -1 with NaN
df_feat[cat_cols_ps] = df_feat[cat_cols_ps].replace(-1, np.nan)
df_feat[num_cols_ps] = df_feat[num_cols_ps].replace(-1, np.nan)

# Impute
for col in cat_cols_ps:
    mode_val = df_feat[col].mode()
    if len(mode_val) > 0:
        df_feat[col] = df_feat[col].fillna(mode_val[0])
    else:
        df_feat[col] = df_feat[col].fillna(0)

for col in num_cols_ps:
    df_feat[col] = df_feat[col].fillna(df_feat[col].median())

print("Missing after imputation:", df_feat.isnull().sum().sum())


In [ ]:
# Encode
ord_enc_ps = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_cat_ps   = ord_enc_ps.fit_transform(
    df_feat[cat_cols_ps].astype(str)).astype(np.int64)

scaler_ps = StandardScaler()
X_num_ps  = scaler_ps.fit_transform(
    df_feat[num_cols_ps].values.astype(np.float32))

cat_cards_ps = [len(cats) for cats in ord_enc_ps.categories_]
print("Cat cardinalities (first 5):", cat_cards_ps[:5])

# Combined array
X_all_ps = np.concatenate([X_num_ps, X_cat_ps.astype(np.float32)], axis=1)
print("X_all shape:", X_all_ps.shape)


## Data Splitting (60 / 20 / 20)

In [ ]:
idx = np.arange(len(y))
idx_tv, idx_test = train_test_split(idx, test_size=TEST_SIZE, random_state=42, stratify=y)
idx_train, idx_val = train_test_split(idx_tv, test_size=VAL_FRAC, random_state=42,
                                       stratify=y[idx_tv])

X_tr_all, X_v_all, X_te_all = X_all_ps[idx_train], X_all_ps[idx_val], X_all_ps[idx_test]
X_tr_num, X_v_num, X_te_num = X_num_ps[idx_train], X_num_ps[idx_val], X_num_ps[idx_test]
X_tr_cat, X_v_cat, X_te_cat = X_cat_ps[idx_train], X_cat_ps[idx_val], X_cat_ps[idx_test]
y_train_ps, y_val_ps, y_test_ps = y[idx_train], y[idx_val], y[idx_test]

sc_ps = StandardScaler()
X_tr_sc = sc_ps.fit_transform(X_tr_all)
X_v_sc  = sc_ps.transform(X_v_all)
X_te_sc = sc_ps.transform(X_te_all)

spw = float((y_train_ps == 0).sum()) / float((y_train_ps == 1).sum())
print(f"Train: {X_tr_sc.shape}, Val: {X_v_sc.shape}, Test: {X_te_sc.shape}")
print(f"scale_pos_weight = {spw:.2f}")


## Helper Functions

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_porto_metrics(y_true, y_pred, y_prob):
    acc  = accuracy_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)
    gini = 2 * auc - 1
    f1   = f1_score(y_true, y_pred, zero_division=0)
    return acc, auc, gini, f1


def train_ft_transformer(model, X_num_tr, X_cat_tr, y_tr,
                          X_num_v, X_cat_v, y_v,
                          lr=1e-3, n_epochs=100, batch_size=256,
                          task='regression', device_='cpu'):
    model = model.to(device_)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss() if task == 'regression' else nn.BCEWithLogitsLoss()

    X_num_tr_t = torch.FloatTensor(X_num_tr).to(device_)
    X_cat_tr_t = torch.LongTensor(X_cat_tr).to(device_) if X_cat_tr is not None else None
    y_tr_t     = torch.FloatTensor(y_tr.astype(np.float32)).to(device_)
    X_num_v_t  = torch.FloatTensor(X_num_v).to(device_)
    X_cat_v_t  = torch.LongTensor(X_cat_v).to(device_) if X_cat_v is not None else None
    y_v_t      = torch.FloatTensor(y_v.astype(np.float32)).to(device_)

    train_losses, val_losses = [], []
    best_val   = float('inf')
    best_state = None
    patience   = 20
    pat_cnt    = 0

    for epoch in range(n_epochs):
        model.train()
        n   = len(X_num_tr_t)
        idx_e = torch.randperm(n)
        ep_loss = 0.0
        for i in range(0, n, batch_size):
            b  = idx_e[i:i+batch_size]
            xn = X_num_tr_t[b]
            xc = X_cat_tr_t[b] if X_cat_tr_t is not None else None
            yb = y_tr_t[b]
            optimizer.zero_grad()
            out  = model(xn, xc).squeeze(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            vout  = model(X_num_v_t, X_cat_v_t).squeeze(-1)
            vloss = criterion(vout, y_v_t).item()
        train_losses.append(ep_loss / n)
        val_losses.append(vloss)
        if vloss < best_val:
            best_val   = vloss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_cnt    = 0
        else:
            pat_cnt += 1
        if pat_cnt >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_losses, val_losses


def predict_ft_transformer(model, X_num, X_cat, device_, batch_size=512):
    model.eval()
    model   = model.to(device_)
    X_num_t = torch.FloatTensor(X_num).to(device_)
    X_cat_t = torch.LongTensor(X_cat).to(device_) if X_cat is not None else None
    preds   = []
    with torch.no_grad():
        for i in range(0, len(X_num_t), batch_size):
            xn  = X_num_t[i:i+batch_size]
            xc  = X_cat_t[i:i+batch_size] if X_cat_t is not None else None
            out = model(xn, xc).squeeze(-1)
            preds.append(out.cpu().numpy())
    return np.concatenate(preds)


## Model 1: TabNet

In [ ]:
all_results = []

def tabnet_porto_objective(trial):
    n_d     = trial.suggest_int('n_d', 8, 64)
    n_a     = trial.suggest_int('n_a', 8, 64)
    n_steps = trial.suggest_int('n_steps', 3, 10)
    gamma   = trial.suggest_float('gamma', 1.0, 2.0)
    lr      = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    set_seed(42)
    m = TabNetClassifier(n_d=n_d, n_a=n_a, n_steps=n_steps, gamma=gamma,
                         optimizer_params={'lr': lr}, verbose=0, seed=42,
                         device_name='cuda' if torch.cuda.is_available() else 'cpu')
    m.fit(X_tr_sc, y_train_ps.astype(int),
          eval_set=[(X_v_sc, y_val_ps.astype(int))],
          eval_name=['val'], eval_metric=['auc'],
          patience=15, max_epochs=100, batch_size=1024, virtual_batch_size=256)
    prob = m.predict_proba(X_v_sc)[:, 1]
    return -roc_auc_score(y_val_ps, prob)

study_tn = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_tn.optimize(tabnet_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_tn = study_tn.best_params
print(f"Best TabNet params: {best_tn}")


In [ ]:
print("Training TabNet across seeds...")
for seed in SEEDS:
    set_seed(seed)
    m = TabNetClassifier(
        n_d=best_tn['n_d'], n_a=best_tn['n_a'], n_steps=best_tn['n_steps'],
        gamma=best_tn['gamma'], optimizer_params={'lr': best_tn['lr']},
        verbose=0, seed=seed,
        device_name='cuda' if torch.cuda.is_available() else 'cpu')
    m.fit(X_tr_sc, y_train_ps.astype(int),
          eval_set=[(X_v_sc, y_val_ps.astype(int))],
          eval_name=['val'], eval_metric=['auc'],
          patience=20, max_epochs=200, batch_size=1024, virtual_batch_size=256)
    preds = m.predict(X_te_sc)
    probs = m.predict_proba(X_te_sc)[:, 1]
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, preds, probs)
    all_results.append({'method': 'TabNet', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Model 2: FT-Transformer

In [ ]:
n_num_ps = X_tr_num.shape[1]

def ft_porto_objective(trial):
    d_token   = trial.suggest_categorical('d_token', [64, 128, 192])
    n_blocks  = trial.suggest_int('n_blocks', 1, 3)
    attn_drop = trial.suggest_float('attention_dropout', 0.0, 0.3)
    ffn_drop  = trial.suggest_float('ffn_dropout', 0.0, 0.3)
    lr        = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    set_seed(42)
    model = rtdl.FTTransformer.make_baseline(
        n_num_features=n_num_ps,
        cat_cardinalities=cat_cards_ps,
        d_token=d_token,
        n_blocks=n_blocks,
        attention_dropout=attn_drop,
        ffn_d_hidden=int(d_token * 4 / 3),
        ffn_dropout=ffn_drop,
        residual_dropout=0.0,
        last_layer_query_idx=[-1],
        d_out=1,
    )
    model, _, _ = train_ft_transformer(
        model, X_tr_num, X_tr_cat, y_train_ps,
        X_v_num, X_v_cat, y_val_ps,
        lr=lr, n_epochs=50, batch_size=256,
        task='classification', device_=str(device))
    raw  = predict_ft_transformer(model, X_v_num, X_v_cat, str(device))
    prob = torch.sigmoid(torch.tensor(raw)).numpy()
    return -roc_auc_score(y_val_ps, prob)

study_ft = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_ft.optimize(ft_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_ft = study_ft.best_params
print(f"Best FT-Transformer params: {best_ft}")


In [ ]:
print("Training FT-Transformer across seeds...")
ft_train_curves = {}
for seed in SEEDS:
    set_seed(seed)
    model = rtdl.FTTransformer.make_baseline(
        n_num_features=n_num_ps,
        cat_cardinalities=cat_cards_ps,
        d_token=best_ft['d_token'],
        n_blocks=best_ft['n_blocks'],
        attention_dropout=best_ft['attention_dropout'],
        ffn_d_hidden=int(best_ft['d_token'] * 4 / 3),
        ffn_dropout=best_ft['ffn_dropout'],
        residual_dropout=0.0,
        last_layer_query_idx=[-1],
        d_out=1,
    )
    model, tr_l, va_l = train_ft_transformer(
        model, X_tr_num, X_tr_cat, y_train_ps,
        X_v_num, X_v_cat, y_val_ps,
        lr=best_ft['lr'], n_epochs=100, batch_size=256,
        task='classification', device_=str(device))
    ft_train_curves[seed] = (tr_l, va_l)
    raw  = predict_ft_transformer(model, X_te_num, X_te_cat, str(device))
    prob = torch.sigmoid(torch.tensor(raw)).numpy()
    pred = (prob >= 0.5).astype(int)
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, pred, prob)
    all_results.append({'method': 'FT-Transformer', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Model 3: XGBoost

In [ ]:
def xgb_porto_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 8),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'scale_pos_weight': spw,
        'random_state':    42
    }
    set_seed(42)
    m = xgb.XGBClassifier(**params, verbosity=0, use_label_encoder=False,
                           eval_metric='auc')
    m.fit(X_tr_all, y_train_ps)
    prob = m.predict_proba(X_v_all)[:, 1]
    return -roc_auc_score(y_val_ps, prob)

study_xgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(xgb_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_xgb = study_xgb.best_params
print(f"Best XGBoost params: {best_xgb}")


In [ ]:
print("Training XGBoost across seeds...")
xgb_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = xgb.XGBClassifier(**best_xgb, scale_pos_weight=spw,
                           random_state=seed, verbosity=0,
                           use_label_encoder=False, eval_metric='auc')
    m.fit(X_tr_all, y_train_ps)
    preds = m.predict(X_te_all)
    probs = m.predict_proba(X_te_all)[:, 1]
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, preds, probs)
    all_results.append({'method': 'XGBoost', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    xgb_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Model 4: LightGBM

In [ ]:
def lgb_porto_objective(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 500),
        'num_leaves':    trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'random_state':  42, 'verbose': -1, 'is_unbalance': True
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr_all, y_train_ps)
    prob = m.predict_proba(X_v_all)[:, 1]
    return -roc_auc_score(y_val_ps, prob)

study_lgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(lgb_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_lgb = study_lgb.best_params
print(f"Best LightGBM params: {best_lgb}")


In [ ]:
print("Training LightGBM across seeds...")
lgb_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = lgb.LGBMClassifier(**best_lgb, random_state=seed, verbose=-1,
                            is_unbalance=True)
    m.fit(X_tr_all, y_train_ps)
    preds = m.predict(X_te_all)
    probs = m.predict_proba(X_te_all)[:, 1]
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, preds, probs)
    all_results.append({'method': 'LightGBM', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    lgb_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Model 5: Random Forest

In [ ]:
def rf_porto_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state':    42
    }
    set_seed(42)
    m = RandomForestClassifier(**params, class_weight='balanced', n_jobs=-1)
    m.fit(X_tr_all, y_train_ps)
    prob = m.predict_proba(X_v_all)[:, 1]
    return -roc_auc_score(y_val_ps, prob)

study_rf = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(rf_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_rf = study_rf.best_params
print(f"Best RF params: {best_rf}")


In [ ]:
print("Training Random Forest across seeds...")
rf_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = RandomForestClassifier(**best_rf, random_state=seed,
                                class_weight='balanced', n_jobs=-1)
    m.fit(X_tr_all, y_train_ps)
    preds = m.predict(X_te_all)
    probs = m.predict_proba(X_te_all)[:, 1]
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, preds, probs)
    all_results.append({'method': 'RandomForest', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    rf_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Results

In [ ]:
df_res = pd.DataFrame(all_results)
summary = df_res.groupby('method').agg(
    acc_mean=('accuracy', 'mean'),  acc_std=('accuracy', 'std'),
    auc_mean=('auc', 'mean'),       auc_std=('auc', 'std'),
    gini_mean=('gini', 'mean'),     gini_std=('gini', 'std'),
    f1_mean=('f1', 'mean'),         f1_std=('f1', 'std')
).round(4)

summary['Accuracy']      = summary['acc_mean'].astype(str)  + ' +/- ' + summary['acc_std'].astype(str)
summary['AUC-ROC']       = summary['auc_mean'].astype(str)  + ' +/- ' + summary['auc_std'].astype(str)
summary['Norm. Gini']    = summary['gini_mean'].astype(str) + ' +/- ' + summary['gini_std'].astype(str)
summary['F1']            = summary['f1_mean'].astype(str)   + ' +/- ' + summary['f1_std'].astype(str)
print(summary[['Accuracy', 'AUC-ROC', 'Norm. Gini', 'F1']])


## Visualizations

In [ ]:
methods = summary.index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(methods, summary['acc_mean'].values, yerr=summary['acc_std'].values,
               capsize=5, color='steelblue', alpha=0.8)
axes[0, 0].set_title('Accuracy')
axes[0, 0].tick_params(axis='x', rotation=30)

axes[0, 1].bar(methods, summary['auc_mean'].values, yerr=summary['auc_std'].values,
               capsize=5, color='darkorange', alpha=0.8)
axes[0, 1].set_title('AUC-ROC')
axes[0, 1].tick_params(axis='x', rotation=30)

axes[1, 0].bar(methods, summary['gini_mean'].values, yerr=summary['gini_std'].values,
               capsize=5, color='forestgreen', alpha=0.8)
axes[1, 0].set_title('Normalized Gini')
axes[1, 0].tick_params(axis='x', rotation=30)

axes[1, 1].bar(methods, summary['f1_mean'].values, yerr=summary['f1_std'].values,
               capsize=5, color='firebrick', alpha=0.8)
axes[1, 1].set_title('F1 Score')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.suptitle('Porto Seguro – Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# FT-Transformer training curves
fig, axes = plt.subplots(1, len(SEEDS), figsize=(15, 4))
for ax, seed in zip(axes, SEEDS):
    tr_l, va_l = ft_train_curves[seed]
    ax.plot(tr_l, label='Train')
    ax.plot(va_l, label='Val')
    ax.set_title(f'Seed {seed}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend()
plt.suptitle('FT-Transformer Training Curves', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# XGBoost feature importance (top 20)
if xgb_model_last is not None:
    fi = xgb_model_last.feature_importances_
    all_feat_names = num_cols_ps + cat_cols_ps
    fi_df = pd.DataFrame({'feature': all_feat_names[:len(fi)], 'importance': fi})
    fi_df = fi_df.sort_values('importance', ascending=True).tail(20)
    fi_df.plot.barh(x='feature', y='importance', figsize=(9, 7),
                    legend=False, color='teal')
    plt.title('XGBoost Top-20 Feature Importance')
    plt.tight_layout()
    plt.show()


In [ ]:
# LightGBM feature importance (top 20)
if lgb_model_last is not None:
    fi = lgb_model_last.feature_importances_
    all_feat_names = num_cols_ps + cat_cols_ps
    fi_df = pd.DataFrame({'feature': all_feat_names[:len(fi)], 'importance': fi})
    fi_df = fi_df.sort_values('importance', ascending=True).tail(20)
    fi_df.plot.barh(x='feature', y='importance', figsize=(9, 7),
                    legend=False, color='mediumpurple')
    plt.title('LightGBM Top-20 Feature Importance')
    plt.tight_layout()
    plt.show()


## Analysis & Conclusions

### Summary
We benchmarked five models on the Porto Seguro safe-driver prediction task.

- The dataset is **severely imbalanced** (~3.6 % positive), so **Normalized Gini**
  (= 2·AUC − 1) is the key competition metric.
- **LightGBM** with `is_unbalance=True` and **XGBoost** with `scale_pos_weight`
  are well-suited for this imbalanced setting.
- **FT-Transformer** uses learned categorical embeddings, which can capture
  non-linear interactions between the many categorical features.
- **TabNet** provides built-in feature selection via sparse attention.
- **Random Forest** with `class_weight='balanced'` offers a strong tree baseline.

### Observations
- Gini scores on **synthetic** data are near 0 because the synthetic labels are
  random (independent of features). Real data results will differ substantially.
- The preprocessing pipeline — replacing -1 sentinels, mode/median imputation,
  OrdinalEncoding — mirrors common Kaggle solutions for this competition.
- 3-seed evaluation reduces variance in reported Gini.

### Next Steps
- Feature engineering: polynomial interactions among `ps_reg` features.
- Calibrated probability outputs for better threshold tuning.
- Stacking / blending of LightGBM + XGBoost predictions.
- SHAP analysis to understand which feature groups drive predictions.
